In [ ]:
import pandas as pd
import numpy as np 
import os
import json

In [ ]:
artifacts_path = "../exp_results/tf-idf_enginering/"

# Create the folder if it doesn't exist
os.makedirs(artifacts_path, exist_ok=True)

existing_experiments = []
if os.path.exists(artifacts_path):
    for folder_name in os.listdir(artifacts_path):
        folder_path = os.path.join(artifacts_path, folder_name)
        if os.path.isdir(folder_path) and folder_name.isdigit():
            existing_experiments.append(int(folder_name))


EXP_NUMBER = max(existing_experiments, default=0) + 1

#create the experience folder
os.makedirs(f"{artifacts_path}/{EXP_NUMBER}")

NUM_DATA_POINTS = 50000

print(f"Current experiment number: {EXP_NUMBER}")

**load the data**

In [ ]:

#read the json file
data_file_path = "../data/processed/multinli_1.0_train_cleaned.csv"
data = pd.read_csv(data_file_path, sep='µ')

#get a random sample of the data
data = data.sample(n = NUM_DATA_POINTS, random_state = 42)
data.head()

### Apply TF-IDF

**set text preprocessing config**

In [ ]:
remove_words_list = []

preprocessing_config = {
    "remove_accents": True,
    "remove_extra_spaces": True,
    "lowercase": True,
    "lemmatization": True,
    "spacy_model": "en_core_web_sm",
    "remove_words_list": remove_words_list,
}

#write the configuration to a json file
with open(f"{artifacts_path}/{EXP_NUMBER}/preprocessing_config.json", "w") as f:
    json.dump(preprocessing_config, f, indent=4)


**text preprocessing**

In [ ]:
import spacy
import re
import unicodedata
from tqdm import tqdm
from nltk.corpus import stopwords
import nltk

# Download stopwords if not already present
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

def clean_text(text, config):
    if config["remove_accents"]:
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    
    text = re.sub(r"[^a-zA-Z\s]", ' ', text)
    
    if config["remove_extra_spaces"]:
        text = re.sub(r'\s+', ' ', text).strip()
    
    if config["lowercase"]:
        text = text.lower()
    
    return text

# Load spacy model (if lemmatization is enabled)
if preprocessing_config["lemmatization"]:
    nlp = spacy.load(
        preprocessing_config["spacy_model"],
        disable=["parser", "ner"]
    )
else:
    nlp = None


#Get stop words to remove
stopwords_to_remove = preprocessing_config['remove_words_list']

def preprocess_texts(texts, batch_size=10000):
    cleaned = []

    texts = [clean_text(t, preprocessing_config) for t in texts]

    if preprocessing_config["lemmatization"] and nlp is not None:
        for doc in tqdm(nlp.pipe(texts, batch_size=batch_size), total=len(texts)):
            tokens = [
                token.lemma_.lower()
                for token in doc
                if token.lemma_.lower() not in stopwords_to_remove
            ]
            cleaned.append(" ".join(tokens))
    else:
        for text in tqdm(texts, total=len(texts)):
            tokens = text.lower().split() if preprocessing_config["lowercase"] else text.split()
            tokens = [
                token
                for token in tokens
                if token not in stopwords_to_remove
            ]
            cleaned.append(" ".join(tokens))

    return cleaned


In [ ]:
preprocess_texts(['This is a sample text! It includes numbers like 123 and special characters like @#$%.'])

In [ ]:
#clean text sentence1 and sentence2
data['sentence1_cleaned'] = preprocess_texts(data['sentence1'].astype(str).tolist())
data['sentence2_cleaned'] = preprocess_texts(data['sentence2'].astype(str).tolist())

**set tf-idf configurations**

In [ ]:
tf_idf_config = {
    "ngram_range": (1, 1),
    "min_df": 0.001,
}

#write the configuration to a json file
with open(f"{artifacts_path}/{EXP_NUMBER}/tf_idf_config.json", "w") as f:
    json.dump(tf_idf_config, f, indent=4)
    

**tf-idf**

In [ ]:
#split before tf-idf to avoid leakage
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

data_train, data_val = train_test_split(
    data,
    test_size=0.1,
    random_state=42,
    stratify=data["gold_label"]
)

# Split train and validation text for TF-IDF fitting
all_sentences_train = data_train["sentence1_cleaned"].tolist() + data_train["sentence2_cleaned"].tolist()

tfidf = TfidfVectorizer(
    min_df=tf_idf_config["min_df"],
    max_df=tf_idf_config["max_df"],
    ngram_range=tf_idf_config["ngram_range"]
)
tfidf.fit(all_sentences_train)

# Transform train and validation using the same vectorizer
tfidf_s1_train = tfidf.transform(data_train["sentence1_cleaned"]).toarray()
tfidf_s2_train = tfidf.transform(data_train["sentence2_cleaned"]).toarray()
tfidf_s1_val = tfidf.transform(data_val["sentence1_cleaned"]).toarray()
tfidf_s2_val = tfidf.transform(data_val["sentence2_cleaned"]).toarray()

In [ ]:
#use the tfidf vectors as features (concatenate sentence1 and sentence2)
feature_names_s1 = [f"s1_{w}" for w in tfidf.get_feature_names_out()]
feature_names_s2 = [f"s2_{w}" for w in tfidf.get_feature_names_out()]
tf_idf_feature = feature_names_s1 + feature_names_s2

features_train = pd.DataFrame(
    np.hstack([tfidf_s1_train, tfidf_s2_train]),
    columns=tf_idf_feature
)
features_val = pd.DataFrame(
    np.hstack([tfidf_s1_val, tfidf_s2_val]),
    columns=tf_idf_feature
)

#add the features to the cleaned_data
data_tf_idf_train = pd.concat([data_train.reset_index(drop=True), features_train.reset_index(drop=True)], axis=1)
data_tf_idf_val = pd.concat([data_val.reset_index(drop=True), features_val.reset_index(drop=True)], axis=1)

#drop sentence1 and sentence2
data_tf_idf_train = data_tf_idf_train.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])
data_tf_idf_val = data_tf_idf_val.drop(columns=['sentence1', 'sentence2', 'genre', 'annotator_labels'])
data_tf_idf_train.head()

In [ ]:
len(tf_idf_feature) / 2

### feature engineering

In [ ]:
#add feature length sentence1 and sentence2
for df in (data_tf_idf_train, data_tf_idf_val):
    df['s1_length'] = df['sentence1_cleaned'].apply(lambda x: len(str(x).split()))
    df['s2_length'] = df['sentence2_cleaned'].apply(lambda x: len(str(x).split()))

In [ ]:
#add feature to detect negation words
negation_words = set(['not', 'no', 'never', 'neither', 'nobody', 'nothing', 'nowhere', 'none', 'nor', 'cannot', "t", 'without', 'hardly', 'scarcely', 'barely', 'seldom', 'rarely'])

#add feature who many negation words in sentence1 and sentence2
def count_negation_words(text):
    words = str(text).lower().split()
    return sum(1 for word in words if word in negation_words)

for df in (data_tf_idf_train, data_tf_idf_val):
    df['s1_negation_count'] = df['sentence1_cleaned'].apply(count_negation_words)
    df['s2_negation_count'] = df['sentence2_cleaned'].apply(count_negation_words)

In [ ]:
#add feature to detect percentage of shared words 
def shared_word_percentage(row):
    s1_words = set(str(row['sentence1_cleaned']).split())
    s2_words = set(str(row['sentence2_cleaned']).split())
    if len(s1_words) == 0 or len(s2_words) == 0:
        return 0.0
    shared_words = s1_words.intersection(s2_words)
    total_words = s1_words.union(s2_words)
    return len(shared_words) / len(total_words) if len(total_words) > 0 else 0.0

for df in (data_tf_idf_train, data_tf_idf_val):
    df['shared_word_percentage_sentence1_sentence2'] = df.apply(shared_word_percentage, axis=1)

In [ ]:
# add feature to detect antonyms between sentence1 and sentence2
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def count_antonyms(row):
    s1_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence1_cleaned']).split()]
    s2_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence2_cleaned']).split()]
    
    antonym_count = 0
    
    for word in s1_words:
        antonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                for ant in lemma.antonyms():  # include all antonyms
                    antonyms.add(lemmatizer.lemmatize(ant.name().lower()))
        
        # Increment if any antonym exists in S2
        if any(ant in s2_words for ant in antonyms):
            antonym_count += 1
            
    return antonym_count

for df in (data_tf_idf_train, data_tf_idf_val):
    df['antonym_count'] = df.apply(count_antonyms, axis=1)

**feature combinaison**

In [ ]:
for df in (data_tf_idf_train, data_tf_idf_val):
    df["ratio_length_sentence1_sentence2"] = df["s1_length"] / (df["s2_length"] + 1) 
    df['ratio_negation_count_sentence1_sentence2'] = df['s1_negation_count'] / (df['s2_negation_count'] + 1)
    df['antonym_count_ratio_sentence1_sentence2'] = df['antonym_count'] / (df["s1_length"] + df["s2_length"])

**keep only required features**

In [ ]:
#keep only feature needed for training
features_to_keep = tf_idf_feature + ["shared_word_percentage_sentence1_sentence2",
                                    "ratio_length_sentence1_sentence2",
                                    "ratio_negation_count_sentence1_sentence2",
                                    "antonym_count_ratio_sentence1_sentence2"
                                ] + ["gold_label"]

data_tf_idf_train = data_tf_idf_train[features_to_keep]
data_tf_idf_val = data_tf_idf_val[features_to_keep]

### train models

In [ ]:
#use the pre-split data for train and validation

#drop columns gold_label
cleaned_data_train = data_tf_idf_train.drop(columns=['gold_label'])
cleaned_data_val = data_tf_idf_val.drop(columns=['gold_label'])

X_train, X_test = cleaned_data_train, cleaned_data_val
y_train, y_test = data_tf_idf_train['gold_label'], data_tf_idf_val['gold_label']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json

models = {
    # "Naive Bayes": GaussianNB(),
    # "KNeighbor Classifier": KNeighborsClassifier(),
    "XGBoost": xgb.XGBClassifier(),
    #"Random Forest": RandomForestClassifier(n_jobs=-1),
    # "Logistic Regression": LogisticRegression(multi_class='multinomial', n_jobs=-1),
    # "SVM": SVC(kernel='linear', decision_function_shape='ovo')
}

# encode labels for XGBoost (needs integer classes)
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)

for model_name, model in models.items():
    print(f"Training {model_name}...")

    if model_name == "XGBoost":
        model.fit(X_train, y_train_enc)
        pred_enc = model.predict(X_test)
        pred = label_encoder.inverse_transform(pred_enc)
        class_names = label_encoder.classes_
    # elif model_name in {'SVM', 'Logistic Regression', 'KNeighbor Classifier'}:
    #     model.fit(X_train_svd, y_train)
    #     pred = model.predict(X_test_svd)
    #     class_names = model.classes_
    # elif model_name in {'Naive Bayes', 'Random Forest'}:
    #     model.fit(X_train, y_train)
    #     pred = model.predict(X_test)
    #     class_names = model.classes_

    # --- per-class metrics as dict ---
    per_class_metrics = {}
    for cls in class_names:
        per_class_metrics[cls] = {
            "precision": float(precision_score(y_test, pred, average=None, labels=[cls])[0]),
            "recall": float(recall_score(y_test, pred, average=None, labels=[cls])[0]),
            "f1": float(f1_score(y_test, pred, average=None, labels=[cls])[0])
        }

    # --- overall metrics ---
    accuracy = accuracy_score(y_test, pred)
    precision_weighted = precision_score(y_test, pred, average='weighted')
    recall_weighted = recall_score(y_test, pred, average='weighted')
    f1_weighted = f1_score(y_test, pred, average='weighted')

    # save metrics json
    metrics = {
        "per_class_metrics": per_class_metrics,
        "accuracy": accuracy,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }
    
    #confusion matrix
    cm = confusion_matrix(y_test, pred, labels=class_names)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'Confusion Matrix for {model_name}')
    plt.savefig(f"../exp_results/tf-idf_enginering/{EXP_NUMBER}/confusion_matrix.png")
    plt.close()
    

    with open(f"../exp_results/tf-idf_enginering/{EXP_NUMBER}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)